# Delta Compression Demo: Full Precision vs 4-bit Quantized

## Setup and Imports

In [1]:
import torch
import time
import sys
from pathlib import Path
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Add the project root to path to import constants
sys.path.append('..')
from constants import BASE_MODEL

## Configuration

Set your model paths here.

In [2]:
# Model paths
base_model_path = BASE_MODEL  # Base model
model_1_path = "experiments/run_final/merged_models/task_0_MeetingBank"  # First fine-tuned model (base + LoRA)

print(f"Base Model: {base_model_path}")
print(f"Model 1: {model_1_path}")

Base Model: openlm-research/open_llama_3b_v2
Model 1: experiments/run_final/merged_models/task_0_MeetingBank


## Helper Functions

In [3]:
def get_quantization_config(bits):
    """
    Get BitsAndBytesConfig for quantization.
    
    Args:
        bits: 4 or 8 for quantization bits
    
    Returns:
        BitsAndBytesConfig for the specified bits
    """
    if bits == 4:
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )
    elif bits == 8:
        return BitsAndBytesConfig(
            load_in_8bit=True,
        )
    else:
        raise ValueError(f"Unsupported bits: {bits}. Only 4 and 8 are supported.")


def load_model(model_path, quantization_bits=None):
    """
    Load a model with optional quantization.
    
    Args:
        model_path: Path to the model
        quantization_bits: None for no quantization, 4 or 8 for quantized loading
    
    Returns:
        Loaded model
    """
    if quantization_bits:
        print(f"  Loading with {quantization_bits}-bit quantization...")
        quantization_config = get_quantization_config(quantization_bits)
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            quantization_config=quantization_config,
            device_map="auto"
        )
    else:
        print(f"  Loading without quantization...")
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )
    return model


def calculate_delta_nonzero_weights(model_a, model_b, verbose=True):
    """
    Calculate the percentage of nonzero weights that differ between two models.
    
    Args:
        model_a: First model
        model_b: Second model
        verbose: Whether to print detailed layer information
    
    Returns:
        dict: Statistics including delta percentage and layer details
    """
    model_a_weights = model_a.state_dict()
    model_b_weights = model_b.state_dict()
    
    # Get all weight parameters
    weight_keys = [
        key for key in model_a_weights.keys()
        if key.endswith(".weight")
    ]
    
    total_nonzero = 0
    total_weights = 0
    layer_stats = []
    
    if verbose:
        print(f"Total weight parameters to compare: {len(weight_keys)}\n")
    
    for key in weight_keys:
        if key not in model_b_weights:
            if verbose:
                print(f"Warning: {key} not found in second model, skipping...")
            continue
        
        start_time = time.time()
        if verbose:
            print(f"Processing: {key}")
        
        weights_a = model_a_weights[key].cpu().float()
        weights_b = model_b_weights[key].cpu().float()
        
        # Calculate weight difference
        weight_difference = torch.sub(weights_a, weights_b)
        nonzero_count = torch.count_nonzero(weight_difference).item()
        total_nonzero += nonzero_count
        total_weights += weights_a.numel()
        
        # Store layer statistics
        layer_percentage = (nonzero_count / weights_a.numel()) * 100 if weights_a.numel() > 0 else 0
        layer_stats.append({
            'layer': key,
            'nonzero': nonzero_count,
            'total': weights_a.numel(),
            'percentage': layer_percentage
        })
        
        # Print statistics for this layer
        if verbose and nonzero_count > 0:
            print(f"  Nonzero differences: {nonzero_count}/{weights_a.numel()} ({layer_percentage:.4f}%)")
            print(f"  Time: {time.time() - start_time:.2f}s\n")
    
    delta_percentage = (float(total_nonzero) / float(total_weights)) * 100 if total_weights > 0 else 0.0
    
    return {
        'delta_percentage': delta_percentage,
        'total_nonzero': total_nonzero,
        'total_weights': total_weights,
        'layer_stats': layer_stats
    }

## Comparison 1: Full Precision (bfloat16)

Load both models without quantization to get the baseline delta.

In [4]:
print("=" * 80)
print("COMPARISON 1: FULL PRECISION (bfloat16)")
print("=" * 80)
print()

# Load base model without quantization
print("Loading Base Model (full precision)...")
start_time = time.time()
base_model_fp = load_model(base_model_path, quantization_bits=None)
print(f"Loaded in {time.time() - start_time:.2f}s\n")

# Load model 1 without quantization
print("Loading Model 1 (full precision)...")
start_time = time.time()
model_1_fp = load_model(model_1_path, quantization_bits=None)
print(f"Loaded in {time.time() - start_time:.2f}s\n")

COMPARISON 1: FULL PRECISION (bfloat16)

Loading Base Model (full precision)...
  Loading without quantization...


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded in 93.14s

Loading Model 1 (full precision)...
  Loading without quantization...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded in 45.31s



In [5]:
# Calculate delta between full precision models
print("=" * 80)
print("Calculating delta between Full Precision Base Model and Model 1...")
print("=" * 80)
print()

fp_results = calculate_delta_nonzero_weights(base_model_fp, model_1_fp, verbose=True)

print("\n" + "=" * 80)
print("RESULTS - Full Precision Comparison")
print("=" * 80)
print(f"Delta percentage: {fp_results['delta_percentage']:.6f}%")
print(f"Total nonzero differences: {fp_results['total_nonzero']:,}")
print(f"Total weights: {fp_results['total_weights']:,}")
print("=" * 80)

Calculating delta between Full Precision Base Model and Model 1...

Total weight parameters to compare: 237

Processing: model.embed_tokens.weight
Processing: model.layers.0.self_attn.q_proj.weight
  Nonzero differences: 4159437/10240000 (40.6195%)
  Time: 0.09s

Processing: model.layers.0.self_attn.k_proj.weight
  Nonzero differences: 4244453/10240000 (41.4497%)
  Time: 0.03s

Processing: model.layers.0.self_attn.v_proj.weight
  Nonzero differences: 7679293/10240000 (74.9931%)
  Time: 0.03s

Processing: model.layers.0.self_attn.o_proj.weight
  Nonzero differences: 8471178/10240000 (82.7263%)
  Time: 0.02s

Processing: model.layers.0.mlp.gate_proj.weight
  Nonzero differences: 18690359/27648000 (67.6011%)
  Time: 0.07s

Processing: model.layers.0.mlp.up_proj.weight
  Nonzero differences: 18762226/27648000 (67.8611%)
  Time: 0.07s

Processing: model.layers.0.mlp.down_proj.weight
  Nonzero differences: 15511965/27648000 (56.1052%)
  Time: 0.08s

Processing: model.layers.0.input_layernorm

In [6]:
# Cleanup full precision models to free memory
del base_model_fp
del model_1_fp
torch.cuda.empty_cache()
print("Full precision models unloaded from memory")

Full precision models unloaded from memory


## Comparison 2: 4-bit Quantized

Load both models with 4-bit quantization to see how quantization affects delta measurements.

In [7]:
print("=" * 80)
print("COMPARISON 2: 4-BIT QUANTIZED")
print("=" * 80)
print()

# Load base model with 4-bit quantization
print("Loading Base Model (4-bit)...")
start_time = time.time()
base_model_4bit = load_model(base_model_path, quantization_bits=4)
print(f"Loaded in {time.time() - start_time:.2f}s\n")

# Load model 1 with 4-bit quantization
print("Loading Model 1 (4-bit)...")
start_time = time.time()
model_1_4bit = load_model(model_1_path, quantization_bits=4)
print(f"Loaded in {time.time() - start_time:.2f}s\n")

COMPARISON 2: 4-BIT QUANTIZED

Loading Base Model (4-bit)...
  Loading with 4-bit quantization...
Loaded in 143.69s

Loading Model 1 (4-bit)...
  Loading with 4-bit quantization...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded in 29.28s



In [8]:
# Calculate delta between 4-bit models
print("=" * 80)
print("Calculating delta between 4-bit Base Model and Model 1...")
print("=" * 80)
print()

quantized_results = calculate_delta_nonzero_weights(base_model_4bit, model_1_4bit, verbose=True)

print("\n" + "=" * 80)
print("RESULTS - 4-bit Quantized Comparison")
print("=" * 80)
print(f"Delta percentage: {quantized_results['delta_percentage']:.6f}%")
print(f"Total nonzero differences: {quantized_results['total_nonzero']:,}")
print(f"Total weights: {quantized_results['total_weights']:,}")
print("=" * 80)

Calculating delta between 4-bit Base Model and Model 1...

Total weight parameters to compare: 237

Processing: model.embed_tokens.weight
Processing: model.layers.0.self_attn.q_proj.weight
  Nonzero differences: 48451/5120000 (0.9463%)
  Time: 0.01s

Processing: model.layers.0.self_attn.k_proj.weight
  Nonzero differences: 50967/5120000 (0.9954%)
  Time: 0.01s

Processing: model.layers.0.self_attn.v_proj.weight
  Nonzero differences: 236667/5120000 (4.6224%)
  Time: 0.00s

Processing: model.layers.0.self_attn.o_proj.weight
  Nonzero differences: 352955/5120000 (6.8937%)
  Time: 0.00s

Processing: model.layers.0.mlp.gate_proj.weight
  Nonzero differences: 427212/13824000 (3.0904%)
  Time: 0.02s

Processing: model.layers.0.mlp.up_proj.weight
  Nonzero differences: 427887/13824000 (3.0952%)
  Time: 0.01s

Processing: model.layers.0.mlp.down_proj.weight
  Nonzero differences: 263006/13824000 (1.9025%)
  Time: 0.01s

Processing: model.layers.0.input_layernorm.weight
Processing: model.layers

In [9]:
# Cleanup 4-bit models to free memory
del base_model_4bit
del model_1_4bit
torch.cuda.empty_cache()
print("4-bit models unloaded from memory")

4-bit models unloaded from memory


## Summary: Comparing Both Results

In [10]:
import pandas as pd

print("\n" + "=" * 80)
print("OVERALL SUMMARY")
print("=" * 80)
print()
print("Comparison 1: Full Precision (bfloat16)")
print(f"  Delta: {fp_results['delta_percentage']:.6f}%")
print(f"  Changed Weights: {fp_results['total_nonzero']:,} / {fp_results['total_weights']:,}")
print()
print("Comparison 2: 4-bit Quantized")
print(f"  Delta: {quantized_results['delta_percentage']:.6f}%")
print(f"  Changed Weights: {quantized_results['total_nonzero']:,} / {quantized_results['total_weights']:,}")
print()
print("Impact of Quantization:")
delta_diff = abs(fp_results['delta_percentage'] - quantized_results['delta_percentage'])
print(f"  Delta difference: {delta_diff:.6f}%")
if fp_results['delta_percentage'] > quantized_results['delta_percentage']:
    print(f"  Full precision shows {delta_diff:.6f}% more delta than quantized")
elif fp_results['delta_percentage'] < quantized_results['delta_percentage']:
    print(f"  Quantized shows {delta_diff:.6f}% more delta than full precision")
else:
    print(f"  Both show approximately the same delta")
print()

# Show top 10 layers for both comparisons
print("=" * 80)
print("TOP 10 LAYERS - Full Precision")
print("-" * 80)
top_10_fp = sorted(fp_results['layer_stats'], key=lambda x: x['percentage'], reverse=True)[:10]
df_fp = pd.DataFrame(top_10_fp)
df_fp['layer_short'] = df_fp['layer'].apply(lambda x: x.split('.')[-2] + '.' + x.split('.')[-1])
for idx, row in df_fp.iterrows():
    print(f"{idx+1:2d}. {row['layer_short']:30s} - {row['percentage']:6.2f}% ({row['nonzero']:,} / {row['total']:,})")

print()
print("=" * 80)
print("TOP 10 LAYERS - 4-bit Quantized")
print("-" * 80)
top_10_quant = sorted(quantized_results['layer_stats'], key=lambda x: x['percentage'], reverse=True)[:10]
df_quant = pd.DataFrame(top_10_quant)
df_quant['layer_short'] = df_quant['layer'].apply(lambda x: x.split('.')[-2] + '.' + x.split('.')[-1])
for idx, row in df_quant.iterrows():
    print(f"{idx+1:2d}. {row['layer_short']:30s} - {row['percentage']:6.2f}% ({row['nonzero']:,} / {row['total']:,})")

print("\n" + "=" * 80)


OVERALL SUMMARY

Comparison 1: Full Precision (bfloat16)
  Delta: 60.623505%
  Changed Weights: 2,077,248,391 / 3,426,473,600

Comparison 2: 4-bit Quantized
  Delta: 2.721192%
  Changed Weights: 49,409,263 / 1,815,721,600

Impact of Quantization:
  Delta difference: 57.902313%
  Full precision shows 57.902313% more delta than quantized

TOP 10 LAYERS - Full Precision
--------------------------------------------------------------------------------
 1. o_proj.weight                  -  82.73% (8,471,178 / 10,240,000)
 2. o_proj.weight                  -  78.60% (8,048,180 / 10,240,000)
 3. gate_proj.weight               -  75.91% (20,988,822 / 27,648,000)
 4. up_proj.weight                 -  75.87% (20,975,436 / 27,648,000)
 5. o_proj.weight                  -  75.56% (7,737,820 / 10,240,000)
 6. gate_proj.weight               -  75.39% (20,842,863 / 27,648,000)
 7. gate_proj.weight               -  75.24% (20,803,716 / 27,648,000)
 8. up_proj.weight                 -  75.08% (20,757,1